# New_Apple_Import Py File

## Uploading New Apple Raw Data

In [ ]:
import os
import datetime as dt
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import time
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime

engine = create_engine(f"sqlite:///habits.db")

# def max_date():

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    max_date_db = pd.read_sql_query(
        text("""SELECT MAX(DATETIME(adr.startDate)) as 'Max_Raw_UTC_Start_Date'
                FROM apple_data_raw adr"""), 
        connection,
        parse_dates=['Max_Raw_UTC_Start_Date'])   


max_date_db["Max_Raw_UTC_Start_Date"] = (
    pd.to_datetime(max_date_db["Max_Raw_UTC_Start_Date"])
    .dt.tz_localize("UTC"))

max_UTC_date_db = max_date_db["Max_Raw_UTC_Start_Date"].iloc[0]

In [ ]:
# def upload_new_raw_apple_data(inital_max_date):

import pandas as pd
import os 
# ETL step of gathering new data from apple health and upploading it almost as it is apple_data_raw

# Defining the subddirectroy where apple docs live
directory = "apple/"

# Initializing variable
apple_file = "apple_health_export_2025-12-14.csv"

# Read in new CSV File, low_memory=False to avoid dtype warnings
apple = pd.read_csv(os.path.join(directory, apple_file), low_memory=False)

# Fitler the source to only my Apple Watch (Could pull in other devices later like iPhone or gamin)
apple = apple[(apple['sourceName'] == "Ryan’s Apple\xa0Watch") | (apple['sourceName'].isna())]

# Strip the "HKWorkoutActivityType" prefix from the workout types
apple["workoutActivityType"] = apple["workoutActivityType"].str.replace("HKWorkoutActivityType", "")

# Drop the UUID column, apple added this since last 7/18 upload
apple.drop(columns=['uuid'], inplace=True)

# Creating a copy of apple dataframe to avoid settingwithcopy warnings when converting datetime
apple_copied = apple.copy()

# Cleaning step helpful for troubleshooting
del apple

# Convert date columns to datetime format including UTC timezone
datetime_cols = ['startDate', 'endDate', 'creationDate']
for col in datetime_cols:
    apple_copied[col] = pd.to_datetime(apple_copied[col], format='%Y-%m-%d %H:%M:%S %z', utc=True)

# Using the exisiting max date in my db to filter to only new data from apple 
apple_copied = apple_copied[apple_copied["startDate"] > max_UTC_date_db] # dt_utc_test  max_UTC_date_db

In [ ]:
# Appending the new raw data to the apple_data_raw table
apple_copied.to_sql(name="apple_data_raw", con=engine, if_exists='append', index=False)

### Testing/Troubleshooting code

In [ ]:
# Little sanity checking step for troubleshooting
min(apple_copied["startDate"])

In [ ]:
# Filler UTC Datetime variable for testing purposes
from datetime import datetime, timezone

dt_utc_test = datetime(2025, 12, 10, tzinfo=timezone.utc)
max_UTC_date_db = dt_utc_test

## Converting Apple Raw Data to Apple workouts data 

In [ ]:
import os
import datetime as dt
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import time
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime

engine = create_engine(f"sqlite:///habits.db")

#def Read_Apple_Workouts(max_UTC_date_db):

# Reading in only the newly updated data from apple_data_raw (using inital max_UTC_date_db prior to uploading new raw data)
with engine.connect() as connection: 
    apple_raw = pd.read_sql_query( 
        text("""SELECT workoutActivityType, startDate, type, maximum, minimum, sum, average, duration, durationUnit 
        FROM apple_data_raw
        WHERE startDate > :max_date"""), # Kinda odd looks like max_UTC_date_db gets carried over into apple_raw (shouldn't matter though
        connection,                     # because we're only fitering to actual workouts)
        params={"max_date": str(max_UTC_date_db)}, # Using the str version of the datetime for filtering
        parse_dates=['startDate']) # Parse date here takes out need for pd.to_datetime below 

apple_raw["startDate"] = apple_raw["startDate"].dt.tz_localize("UTC")

# Filters df to include only rows where at least one of the workout-related columns is not missing (NaN).
# .notna() Check for Non-Nulls in selected columns 
# .any(axis=1) aggregates rows where there's at least one non-missing value 
apple_workouts = apple_raw[apple_raw.notna().any(axis=1)] 

In [ ]:
#def aw_to_long(apple_workouts):

# Reshape the apple_workouts DataFrame from wide format (one row per workout with multiple measurement columns) to long format (one row per measurement per workout).
df_long = pd.melt(apple_workouts, 
                    id_vars=['startDate', 'workoutActivityType', 'type', 'duration', 'durationUnit'],
                    value_vars=['maximum', 'minimum', 'sum', 'average'], # turned into rows via where the name goes into 'measurement_type' and actual values to the 'value' column
                    var_name='measurement_type', 
                    value_name='value')

# Removing rows where either type or value are null (no meaningful data here)
df_long = df_long[(df_long['type'].notna()) | (df_long['value'].notna())]

# Handle duration rows separately and combine
duration_rows = apple_workouts[apple_workouts['duration'].notna() & apple_workouts['type'].isna()].copy()
duration_rows = duration_rows.assign(
    measurement_type='total',
    value=duration_rows['duration'],
    type='Duration')[['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']]

# Combine and clean
aw_long = pd.concat([df_long[df_long['value'].notna()][['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']],
    duration_rows], ignore_index=True)

# Rename columns for clarity
aw_long.rename(columns={'startDate': 'StartDate', 'workoutActivityType': 'activity', 'type': 'metric', # Why do I rename StartDate to be different than apple data raw?
                       'measurement_type': 'measurement_type', 'value': 'value', 'durationUnit': 'd_unit'}, inplace=True)

# cleaning value column to only go out 2 decimal places
aw_long['value'] = aw_long['value'].round(2)

aw_long.sort_values('StartDate', ascending=False, inplace=True)

In [ ]:
def max_workout_id():
        #Gathering the max workout_id from existing apple_workouts table
        with engine.connect() as connection:
                max_workout_id_df = pd.read_sql_query(
                        text("""SELECT max(workout_id) as workout_id 
                                FROM apple_workouts
                                WHERE activity IS NOT NULL"""), # Need to filter on activity because otherwise workout_id is NA
                        connection)  

                # Grabbing singlular max workout_id from dataset to filter new data off of  
                max_workout_id = int(max_workout_id_df['workout_id'][0]) # Maybe make this an INT variable?

        return max_workout_id

#max_workout_id = max_workout_id()

In [ ]:
#def add_workout_id(aw_long):

# We're able to easily get unique workouts by date because aw_long only has activity filled for the duration metric which every workout has 1 of 
activity_map = aw_long[aw_long['activity'].notnull()][["StartDate", "activity"]]

# Sorting values by date and resetting index (easy way to grab a unique id so long as I'm sorting correctly)
activity_map = activity_map.sort_values('StartDate').reset_index(drop=True)

# Creating a new variable based on the index  
# Previous null values probably from Garmin are keeping workout_id from being an integer in aw_final
activity_map['workout_id'] = activity_map.index.astype(int) + max_workout_id()  # Start IDs from 1

# Merge activity back onto the main dataframe using startDate
aw_final = pd.merge(aw_long, activity_map, on='StartDate', how='left', suffixes=('', '_Specifier'))

# Drop the original activity column
aw_final.drop(columns=['activity'], inplace=True)

# Rename the helper column to activity since that's actually what we'll want
aw_final.rename(columns={'activity_Specifier': 'activity'}, inplace=True)

# Adding new activity type variable for minutes calculation 
aw_final['activity_type'] = np.where(aw_final['activity'].isin(['Running', 'Cycling', 'Swimming', 'Walking']), 'Cardio',
    np.where(aw_final['activity'] == 'TraditionalStrengthTraining', 'Weights', None))

In [ ]:
#def final_upload(aw_final):
aw_final.to_sql(name="apple_workouts", con=engine, if_exists='append', index=False)

# Data Cleaning + Visualization PY

## Reading in Apple Workouts db from SQLite

In [4]:
import os
import datetime as dt
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import time
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime
import plotly.express as px

# Read in cleaned Apple Workouts data 
#def Read_Apple_Workouts():

engine = create_engine(f"sqlite:///habits.db")

# Read in the data and parse datetime columns
with engine.connect() as connection:
    aw_all = pd.read_sql_query(
        text("""SELECT * FROM apple_workouts"""),
        connection)
        #dtype={"workout_id": "Int64"}) # Converting workout_id to Int64 upon entry, will need again if workout_id includes blanks
    
# Converting the string datetime variable from db to with timezone normalization to UTC then converting to EST
# This is needed because of combination of EST and EDT in the data 
# I could do this with a parse dates function in pd.read_sql_query() function above
date_cols = ["StartDate"]
for col in date_cols:
    aw_all[col] = (
        pd.to_datetime(aw_all[col])
        .dt.tz_localize("UTC")
        .dt.tz_convert("US/Eastern"))

### Adding in categorical variables for month and week

In [5]:
# Creating a month categorical variable
# Create a string label for display
aw_all['month'] =  aw_all['StartDate'].dt.strftime('%b %Y') # Need .dt for series/panda 

# Certainly 1 way to get month from datetime but keep it as a datetime variable lol
aw_all['month_date'] =  pd.to_datetime(aw_all['month'], format='%b %Y') # Need .dt for series/panda

# Need to localize datetime to US/Eastern
aw_all['month_date'] = aw_all['month_date'].dt.tz_localize("US/Eastern")

# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = aw_all.sort_values('month_date')['month'].unique()
aw_all['month'] = pd.Categorical(aw_all['month'], categories=month_order, ordered=True)


# Creating a week categorical variable
# calculating out necessary time metrics when reading in apple workouts data
## calculates how many days each weekday is from monday and subtracts that from the orignal date to get back to start of the week Monday
aw_all['week_date'] = (
    aw_all['StartDate'] - pd.to_timedelta(aw_all['StartDate'].dt.weekday, unit="D") # dt.weekday uses Monday as 0 
).dt.normalize() # Resulting date is in EST (I believe this is the correct move when filtering later on)

    # Create a string label for display
aw_all['week'] = aw_all['week_date'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = aw_all.sort_values('week_date')['week'].unique()
aw_all['week'] = pd.Categorical(aw_all['week'], categories=week_order, ordered=True)

## Anlysis

### Filling missing combinations function

In [ ]:
from itertools import product
# Function that fills in missing data for each grouped by df 
def fill_missing_combinations(
    original_df,
    aggregated_df,
    time_col,
    category_col,
    value_cols,
    time_filter=None,
    category_values=None):

    # Step 1: Filter time periods if needed
    if time_filter:
        time_values = original_df[time_filter(original_df)].loc[:, time_col].unique()
    else:
        time_values = original_df[time_col].unique()
    # Step 2: Get all categories
    if category_values:
        category_values = category_values
    else:
        category_values = aggregated_df[category_col].unique()
    # Step 3: Create full index
    full_index = pd.DataFrame(
        product(category_values, time_values),
        columns=[category_col, time_col])
    # Step 4: Merge with aggregated data
    filled_df = pd.merge(full_index, aggregated_df, on=[category_col, time_col], how='left')
    # Step 5: Fill NaNs with 0 for specified value columns
    for col in value_cols:
        filled_df[col] = filled_df[col].fillna(0)

    return filled_df.sort_values(by=[time_col, category_col]).reset_index(drop=True)

### Establishing consistent time filters

In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
from zoneinfo import ZoneInfo
from plotnine import ggplot, aes, geom_bar, geom_text, position_dodge, scale_fill_brewer, scale_y_continuous, labs, theme_seaborn, theme, scale_fill_manual, element_line, geom_hline, geom_line, geom_point, scale_x_datetime, element_text, element_blank, geom_boxplot, stat_summary

today = datetime.now(tz=ZoneInfo("US/Eastern"))

l_3_m = today - relativedelta(weeks=14)
l_7_m = today - relativedelta(months=7)
l_1_y = today - relativedelta(years=1)

### DF and Visualizations pairs

#### Frequency bar chart grouped by exercise type and month

In [ ]:
# def gen_month_freq_df(aw_all, filter1):
activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

df_counts = (aw_all[(aw_all['metric'] == 'Duration') & (aw_all['activity'].isin(activities)) & 
                    (aw_all['month_date'] > l_7_m)]
                    .groupby(['month_date', 'activity'], observed=True) # Include only combinations that actually occur in the data
                    .size() 
                    .reset_index(name='n'))

month_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='month_date',
    category_col='activity',
    value_cols=['n'])

month_count_data['n'] = month_count_data['n'].astype(int)  # Convert to int for better readability

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
month_count_data['activity'] = month_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

# Merging in categorical month label
month_count_data = pd.merge(month_count_data,
    aw_all[['month', 'month_date']].drop_duplicates(),
    on='month_date',
    how='left')

In [ ]:
############### Frequency bar chart grouped by exercise type and month ###############
#month_count_data = gen_month_freq_df(aw_final, l_7_m)

#def Monthly_Freq_BarChart():
    
plot = (ggplot(month_count_data, aes(x='month', y='n', fill='activity')) + 
    geom_bar(stat='identity', position='dodge', color = "Black") +
    
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    
    scale_fill_brewer(type='qual', palette='Set2') +
    scale_y_continuous(breaks = range(0, 14, 2),
                    limits = [0, 13]) +

    labs(title='Workout Frequency by Month and Activity',
        x='',
        y='Sessions',
        fill='Activity',
        color='Goal') +
    #theme_matplotlib() +
    theme_seaborn() +
    theme(figure_size=(10, 5)))

plot

#### Frequency bar chart grouped by exercise type and week

In [ ]:
############### Frequency bar chart grouped by exercise type and week ###############
# def gen_week_freq_df(aw_final, filter2):

activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

df_counts = (aw_all[(aw_all['metric'] == 'Duration') & (aw_all['activity'].isin(activities)) & (aw_all['week_date'] > l_3_m)]
    .groupby(['week_date', 'activity'])
    .size()
    .reset_index(name='n'))

week_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='week_date',
    category_col='activity',
    value_cols=['n'])

week_count_data['n'] = week_count_data['n'].astype(int)  # Convert to int for better readability

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
week_count_data['activity'] = week_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

week_count_data = pd.merge(week_count_data,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Frequency bar chart grouped by exercise type and week ###############
#week_count_data = gen_week_freq_df(aw_final, l_3_m)
#def Weekly_Freq_BarChart():

plot = (ggplot(week_count_data, aes(x='week', y='n', fill='activity')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    
    scale_fill_brewer(type='qual', palette='Set2') +
    #scale_color_manual(values={'Running': 'Black', 'Cycling': 'Gray'}) +
    scale_y_continuous(breaks = range(0, 6),
                    limits = [0, 5]) +

    labs(title='Workout Frequency by Week and Activity',
        x='',
        y='Sessions',
        fill='Activity') +
    #theme_matplotlib() +
    theme_seaborn() +
    theme(figure_size=(10, 5)))

plot

#### Distance per week grouped by exercise type

In [ ]:
############### Distance per week grouped by exercise type ###############
# def gen_distance_df(aw_final, filter2):

miles_week = (aw_all[(aw_all['activity'].isin(['Running', 'Cycling'])) & (aw_all['metric'].str.contains("Distance")) & (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_Miles='sum', n='count')  # compute both mean and count
    .round(2) # Round to 2 decimal places
    .reset_index())

# What's interesting is that if biking or running has 0 observations missing observations won't be filled in
full_miles_week = fill_missing_combinations(
    original_df=miles_week,
    aggregated_df=miles_week,
    time_col='week_date',
    category_col='activity',
    value_cols=['Total_Miles', 'n'])


full_miles_week = pd.merge(full_miles_week,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Distance per week grouped by exercise type ###############
#full_miles_week = gen_distance_df(aw_final, l_3_m)
#def Distance_BarChart():

plot = (ggplot(full_miles_week, aes(x='week', y='Total_Miles', fill='activity')) +
geom_bar(stat='identity', position='dodge', color = "Black") +
geom_text(aes(label='Total_Miles'), position=position_dodge(width=.9), va='bottom') +

scale_y_continuous(breaks = range(0, 36, 5),
                #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                limits = [0, 36]) +

scale_fill_manual(values={'Running': '#a259d9',   
    'Cycling': '#ff9800'}) +

labs(title= "Cardio Miles per Week",
    x="",
    y="Miles",
    fill = "Activity") +
theme_seaborn() +
theme(figure_size=(10, 5),
    panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))

plot

#### Minutes per week grouped by cardio and weights 

In [ ]:
############### Minutes per week grouped by cardio and weights ###############
# def gen_mins_df(aw_final, filter2):

mins_week = (aw_all[
        (aw_all['activity_type'].notna()) &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity_type', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())

full_mins_week = fill_missing_combinations(
    original_df=mins_week,
    aggregated_df=mins_week,
    time_col='week_date',
    category_col='activity_type',
    value_cols=['Total_min', 'n'])

full_mins_week = pd.merge(full_mins_week,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Minutes per week grouped by cardio and weights ###############
#full_mins_week = gen_mins_df(aw_final, l_3_m)
#def Minutes_BarChart():
plot = (ggplot(full_mins_week, aes(x='week', y='Total_min', fill='activity_type')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='Total_min'), format_string='{:.1f}',position=position_dodge(width=.9), va='bottom') + #Format count?

    geom_hline(yintercept=150, color="#549f74", linetype='dashed', size=1) + # Cardio goal
    geom_hline(yintercept=80, color="#b36a62", linetype='dashed', size=1) + # Weights goal
    
    scale_y_continuous(breaks = range(0, 450, 50),
                    #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, 400]) +

    # Manual color scales
    scale_fill_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +  # Bar colors
    
    labs(title= "Minutes per Week by Activity",
        x="",
        y="Minutes", 
        fill="Activity") +
    
    theme_seaborn() +
    theme(figure_size=(10, 5),
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))

plot

#### Minutes per week for all exercises (line grpah - Deprecated)

In [ ]:
############### Minutes per week for all exercises ############### 
# def gen_workout_time_df(aw_final, filter1):

workout_time = (
    aw_all[(aw_all['week_date'] > l_3_m) & (aw_all['metric'] == 'Duration')]
    .groupby('week_date', observed=True)['value']
    .agg(Time='sum', n='count')
    .round(2)
    .reset_index())

workout_time = pd.merge(workout_time,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Minutes per week for all exercises ############### 
#workout_time = gen_workout_time_df(aw_final, l_7_m)
#def Minutes_LineGraph():

plot = (ggplot(workout_time, aes(x='week', y='Time')) +
    geom_line(color = "blue", size = 1) +
    #geom_point(aes(size = "n"), alpha = 0.6, color = "blue") +
    #geom_text(aes(label='n'), format_string='{:.0f}', va='bottom') +
    
    labs(title= "Minutes per Week by Activity",
         x="Week",
         y="Minutes") +

    #scale_x_datetime(date_labels='%b %d', date_breaks='1 week') +
    
    theme_seaborn() +
    
    theme(panel_grid_minor_y = element_line(color = "gray", linetype = "dotted"),
        figure_size=(10, 5),
        axis_text_x=element_text(angle=25, hjust=1),
        legend_position='none',
        axis_ticks_minor_x=element_blank()))

plot

#### Activity Treemap Distribution

In [ ]:
############### Activity Treemap ###############  
#def gen_activity_treemap_df(aw_all, l_3_m):

activites = ['Running', 'Cycling', 'TraditionalStrengthTraining', 'Swimming']

activity_distribution = (aw_all[
        (aw_all['metric'] == "Duration") & 
        (aw_all['activity'].isin(activites)) & 
        (aw_all['week_date'] >= l_3_m)] # Filtering by the last 3 months
    .groupby(['activity'])['value']
    .agg(n='count')
    .reset_index())
            
# Add percent of total column
total = activity_distribution['n'].sum()
activity_distribution['percent'] = activity_distribution['n'] / total

# Format labels as "Activity<br>Count (Percent)"
activity_distribution['label'] = activity_distribution.apply(lambda row: f"{row['activity']}<br>{row['n']} ({row['percent']:.1%})", axis=1)

In [ ]:
# Create treemap
#activity_distribution = gen_activity_treemap_df(aw_final, l_3_m)
#def activity_treemap():

fig = px.treemap(
    activity_distribution,
    path=['label'],         # use custom label for full text
    values='n',
    color='n',
    color_continuous_scale='Blues')

fig.update_traces(hovertemplate='%{label}')  
fig.update_layout(showlegend=False, coloraxis_showscale=False)

# treemap_plotly_html = pio.to_html(fig, full_html=False)

#### Step count grouped by month boxplot

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
engine = create_engine(f"sqlite:///habits.db")

#def gen_steps_month_df(filter3):

# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    apple_steps = pd.read_sql_query(
        text("""SELECT type, sum(value) as 'value', date(startDate, 'localtime') as 'date', DATETIME(startDate, 'start of month') as 'month_date'
                FROM apple_data_raw
                WHERE type = 'StepCount' AND value IS NOT NULL and date(startDate, 'start of month','+1 month','-1 day') >= :t_filter
                GROUP BY date(startDate)
                ORDER BY 'date'"""), connection,
        dtype={"value": "Int64"},
        params={"t_filter": l_1_y.astimezone(ZoneInfo("UTC"))},
        parse_dates=['date', 'month_date'])

# Localizing date and month_date to US/Eastern because already convert to localtime in the SQL query
apple_steps['date'] = apple_steps['date'].dt.tz_localize("US/Eastern")
apple_steps['month_date'] = apple_steps['month_date'].dt.tz_localize("US/Eastern")

# Creating a month categorical variable
apple_steps['month'] = apple_steps['date'].dt.strftime('%b %Y') # Need .dt for series/panda

# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = apple_steps.sort_values('month_date')['month'].unique()
apple_steps['month'] = pd.Categorical(apple_steps['month'], categories=month_order, ordered=True)

# apple_steps = gen_steps_month_df(l_1_y)

In [ ]:
############### Step count grouped by month ############### 
#steps_day = gen_steps_month_df(l_1_y)

#def Steps_Boxplot():
plot = (
    ggplot(apple_steps, aes(x='month', y='value', fill='month')) +

    geom_boxplot(color="black") +
    stat_summary(fun_data="mean_cl_boot", geom = "point", fill = "white", color = "red") +

    labs(title="Daily Steps Grouped by Month", 
        x="", 
        y="Steps") +

    scale_y_continuous(breaks = range(0, 32500, 5000),
                    minor_breaks=range(0, 32500, 2500),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, 30000]) +

    theme_seaborn() +

    theme(axis_text_x=element_text(),
        figure_size=(10, 5),
        legend_position='none',
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))
plot

### HTML Tables

#### Filtered Exercise Table

In [ ]:
# Generates a table for the dynamic exercise filter page 
#def specific_exercise_filter(specific_exercise):

specific_exercise = "Chest press machine"  # Simulating user input
columns = ['entry_id', '10RM', 'comment', 'effort', 'reps', 'sets', 'weight', 'workout_type']
workout_df_total = pd.DataFrame(columns=columns)

with engine.connect() as connection:

    entry_ids = pd.read_sql_query(text("SELECT ha.entry_id FROM habit_answers ha WHERE answer = :exercise"), connection, params={"exercise": specific_exercise})

    entry_ids = entry_ids['entry_id'].tolist()


    for id in entry_ids:
        workout_df = pd.read_sql_query(text("SELECT entry_id, question, answer FROM habit_answers ha WHERE entry_id = :id"), connection, params={"id": id})

        workout_df = workout_df.pivot(index='entry_id', columns='question', values='answer').reset_index() 

        workout_df_total = pd.concat([workout_df_total, workout_df], ignore_index=True)

    habit_entries = pd.read_sql_query(text("SELECT log_date, id FROM habit_entries"), connection)

    # Merging and selecting relevant columns
    workout_df_total = pd.merge(workout_df_total, habit_entries, left_on='entry_id', right_on='id', how='left')[["log_date", "workout_type", "weight", "sets", "reps", "effort", "comment"]]

    workout_df_total['comment'] = workout_df_total['comment'].str.replace("nan", "")

    workout_df_total.rename(columns={'log_date': 'Timestamp', 'workout_type': 'Exercise', 'weight': 'Weight', 'sets': 'Sets', 'reps': 'Reps',
                             'effort': 'Effort Level', 'comment': 'Notes:'}, inplace=True)


    # reading in 10RM workouts to add 
    ten_rm_additions = pd.read_sql_query(
        text("""SELECT tc.completion_date as Timestamp, tp.exercise_name as Exercise, tp.target_weight as Weight, tp.sets as Sets, tp.reps as Reps, tc.notes as 'Notes:' 
                FROM tenrm_completions tc
                LEFT JOIN tenrm_plans tp 
                    ON tp.id = tc.plan_id
                WHERE tp.exercise_name = :exercise
                AND tc."timestamp" = (
                        SELECT MAX(tc2."timestamp")
                        FROM tenrm_completions tc2
                        -- Correlated Subquery
                        -- This works because we loop through plan_ids in orig table until it equals max plan_id
                        WHERE tc2.plan_id = tc.plan_id)"""), connection, params={"exercise": specific_exercise})

    # Adding in effort level as a blank variable since 10rm data doesn't track that
    ten_rm_additions['Effort Level'] = ""

    # Combining original workouts with 10rm workouts (Since they have the same columns and format) 
    combined_works = pd.concat([workout_df_total, ten_rm_additions], ignore_index=True) 

    # Convert the timestamp variable to a datetime format ( Could have done this sql query with parse dates too)
    combined_works['Timestamp'] = pd.to_datetime(combined_works['Timestamp'])

    # Sort the combined_works table by date before converting to string output (for readability)
    combined_works = combined_works.sort_values('Timestamp')

    # Converting Timestamp into readable string format (flexability to display time however I want
    combined_works['Timestamp'] = combined_works['Timestamp'].dt.strftime('%B %d, %Y')

#return combined_works.to_html(classes='workout-table', index=False, border=1)

### KPI Calculations

In [ ]:
############### KPI Statisitcs Functions ###############

# Going to have to recreate this function with apple workout data
#def get_kpi_stats(aw_all, apple_steps):

# gathering some date statistics for filtering but I'm not sure I like this method :/
today = datetime.now(tz=ZoneInfo("US/Eastern"))

this_month = today.strftime('%b %Y')
last_month = (today - relativedelta(months=1)).strftime('%b %Y')
current_year = int(today.strftime('%Y'))
duration_length = 10

# Gathering wokrout counts (classifying a true workout to be greater than 10 minutes long)
## Current Month Workout Count
wokrout_count_CM = len(aw_all[(aw_all['month'] == this_month) & (aw_all['metric'] == 'Duration') & (aw_all['value'] >= duration_length)])
## Last month Workout Count
workout_count_LM = len(aw_all[(aw_all['month'] == last_month) & (aw_all['metric'] == 'Duration') & (aw_all['value'] >= duration_length)])
## Current Year Workout Count
workout_count_year = len(aw_all[(aw_all['StartDate'].dt.year == current_year) & (aw_all['metric'] == 'Duration') & (aw_all['value'] >= duration_length)])

# Average daily step count last 3 months
steps_L3_mon = apple_steps[apple_steps['date'] >= l_3_m]['value'].mean().round(0).astype("int")

# Gather average weekly time spent working out using last 3 months data
workout_time_df = aw_all[(aw_all['metric'] == 'Duration') & (aw_all['week_date'] > l_3_m) ].groupby(['week_date'])['value'].agg(Time='sum', n='count').reset_index()
workout_time_hrs_avg = (workout_time_df["Time"].mean()/60).round(2)

# Gather names for above statistic
current_month_name = today.strftime('%B')
last_month_name = (today - relativedelta(months=1)).strftime('%B')

#return workout_count_year, workout_count_LM, wokrout_count_CM, current_month_name, last_month_name, workout_time_hrs_avg, steps_L3_mon

# get_kpi_stats(aw_all, apple_steps)

## Options list (workouts and books)

In [ ]:
# List to log existing workouts from db
def load_workout_options(category):

    with engine.connect() as connection:
        workouts = pd.read_sql_query(
            text("""SELECT DISTINCT answer FROM habit_answers
                    WHERE question = 'workout_type'
                    ORDER BY answer"""), connection)
        
    workouts_list = workouts['answer'].tolist()

    # Adding some logic to use this function multiple places depending on whether I need an "Other" option or not 
    if category == 0:
        pass
    elif category == 1:
        # Add "Other" option to the list of book titles and works. This way other is not saved to the database
        workouts_list.append("Other")        


    return workouts_list

In [ ]:
# Update the list of book titles table in my database with any potential new entries
def load_book_options():

    # read_sql_query simple 
    with engine.connect() as connection:
        books_options = pd.read_sql_query(
            text("""SELECT DISTINCT answer FROM habit_answers
                    WHERE question = 'book_title'
                    ORDER BY answer"""), connection)
        
    book_titles = books_options["answer"].to_list()

    # Add an "Other" option to the list of book titles because 'other' is not saved to the database
    book_titles.append("Other") 

    return book_titles

# DB.py

## Creating DB connection and modifiers

In [ ]:
# db.py
from sqlalchemy import create_engine, event, text

# For SQLite file-based DB
DATABASE_URL = "sqlite:///habits.db"

# Create a single shared engine
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,   # ensures dead connections are detected/recycled
    connect_args={"check_same_thread": False}  # needed for SQLite in multi-threaded apps
)

# Enable WAL mode on every new connection
@event.listens_for(engine, "connect")
def set_sqlite_pragma(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute("PRAGMA journal_mode=WAL")   # Enable Write-Ahead Logging
    cursor.execute("PRAGMA synchronous=FULL") # balance durability vs performance 
    cursor.execute("PRAGMA optimize") 
    cursor.close()


## Creating sqlite tables for app

In [ ]:
# initalizing habits websites with sql tables needed to support it
# I've already gone through this and changed types to be sqlite compliant
def init_db():
    with engine.begin() as conn:

        # habits table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habits (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            name TEXT UNIQUE NOT NULL);"""))

        # habit_entries table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_entries (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            habit_id INTEGER NOT NULL,
                            log_date TEXT NOT NULL,
                            timestamp TEXT NOT NULL,
                            FOREIGN KEY (habit_id) REFERENCES habits(id));"""))

        # habit_answers table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_answers (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            entry_id INTEGER NOT NULL,
                            question TEXT NOT NULL,
                            answer TEXT NOT NULL,
                            FOREIGN KEY (entry_id) REFERENCES habit_entries(id));"""))
        
        # access log table (for IP address logging)
        conn.execute(text("""CREATE TABLE IF NOT EXISTS access_log (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            ip TEXT NOT NULL,
                            endpoint TEXT NOT NULL,
                            timestamp TEXT NOT NULL);"""))
        
        # 10RM workout plans table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS tenrm_plans (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            workout_type TEXT NOT NULL,
                            week_number INTEGER NOT NULL,
                            exercise_name TEXT NOT NULL,
                            target_weight INTEGER NOT NULL,
                            sets INTEGER NOT NULL,
                            reps INTEGER NOT NULL,
                            created_date TEXT NOT NULL,
                            UNIQUE(workout_type, week_number, exercise_name));"""))
        
        # 10RM workout completions table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS tenrm_completions (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            plan_id INTEGER NOT NULL,
                            completion_date TEXT NOT NULL,
                            completed INTEGER NOT NULL,
                            notes TEXT,
                            timestamp TEXT NOT NULL,
                            FOREIGN KEY (plan_id) REFERENCES tenrm_plans(id));"""))

        # apple_data_raw table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS apple_data_raw (
                            type TEXT, 
                            "sourceName" TEXT, 
                            value TEXT, 
                            unit TEXT, 
                            "startDate" TEXT, 
                            "endDate" TEXT, 
                            "creationDate" TEXT, 
                            "sourceVersion" TEXT, 
                            "appleStandHours" REAL, 
                            "appleExerciseTimeGoal" REAL, 
                            bpm REAL, 
                            maximum REAL, 
                            "sum" REAL, 
                            "appleMoveTimeGoal" REAL, 
                            "average" REAL, 
                            time TEXT, 
                            "key" TEXT, 
                            duration REAL, 
                            "dateComponents" TEXT, 
                            "CardioFitnessMedicationsUse" TEXT, 
                            "activeEnergyBurned" REAL, 
                            "appleMoveTime" REAL, 
                            date TEXT, 
                            "activeEnergyBurnedUnit" TEXT, 
                            locale TEXT, 
                            "appleStandHoursGoal" REAL, 
                            "BiologicalSex" TEXT, 
                            "FitzpatrickSkinType" TEXT, 
                            "BloodType" TEXT, 
                            "workoutActivityType" TEXT, 
                            "minimum" REAL, 
                            path TEXT, 
                            "appleExerciseTime" REAL, 
                            "durationUnit" TEXT, 
                            "DateOfBirth" TEXT, 
                            device TEXT, 
                            "activeEnergyBurnedGoal" REAL)"""))


        # Cleaned Apple Workouts table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS apple_workouts (
                            "StartDate" TEXT, 
                            activity TEXT, 
                            metric TEXT, 
                            measurement_type TEXT, 
                            value REAL, 
                            d_unit TEXT, 
                            activity_type TEXT, 
                            workout_id REAL)"""))

# Main.py

In [ ]:
########### Log users IP address after every made request
def get_client_ip():
    """Helper to get the client IP address, accounting for proxies."""
    if request.headers.get("X-Forwarded-For"):
        # Might contain multiple IPs, take the first one
        return request.headers.get("X-Forwarded-For").split(",")[0].strip()
    return request.remote_addr or "unknown"

@app.before_request
def log_ip():
    ip = get_client_ip()
    endpoint = request.path
    today = datetime.now(tz=ZoneInfo("US/Eastern")).strftime("%Y-%m-%d %H:%M:%S")

    with engine.begin() as conn:
        conn.execute(
            text("""INSERT INTO access_log (ip, endpoint, timestamp)
                    VALUES (:ip, :endpoint, :timestamp)"""),
            {"ip": ip, "endpoint": endpoint, "timestamp": today},
        )


## 10RM Functions

Debating if I want to rework how these 10RM functions are written

In [ ]:
########### 10 RM workouts page ###########
@app.route('/10rm_tracker', methods=['GET'])
def tenrm_tracker():
    
    with engine.begin() as conn:
        # Get all workout plans with their latest completion status
        results = conn.execute(text("""
            SELECT 
                tp.id,
                tp.workout_type,
                tp.week_number,
                tp.exercise_name,
                tp.target_weight,
                tp.sets,
                tp.reps,
                tc.completion_date,
                tc.completed,
                tc.notes
            FROM tenrm_plans tp
            LEFT JOIN tenrm_completions tc ON tp.id = tc.plan_id
            AND tc.id = (
                SELECT MAX(id) 
                FROM tenrm_completions 
                WHERE plan_id = tp.id
            )
            ORDER BY tp.workout_type, tp.week_number, tp.exercise_name""")).fetchall()
        
        # Organize data by workout type and week
        organized_data = {}
        for row in results:
            workout_type = row[1]
            week_number = row[2]
            
            if workout_type not in organized_data:
                organized_data[workout_type] = {}
            
            if week_number not in organized_data[workout_type]:
                organized_data[workout_type][week_number] = []
            
            organized_data[workout_type][week_number].append({
                'plan_id': row[0],
                'exercise_name': row[3],
                'target_weight': row[4],
                'sets': row[5],
                'reps': row[6],
                'completion_date': row[7],
                'completed': row[8],
                'notes': row[9]
            })
    
    return render_template('10rm_tracker.html', workout_data=organized_data)

In [ ]:
### I'm not quite sure what this does because 10rm plans is done manually ###
# I don't think this code does anything lol
@app.route('/add_10rm_plan', methods=['POST'])
def add_10rm_plan():
    """Add a new 10RM workout plan for a specific week"""
    try:
        data = request.json
        workout_type = data.get('workout_type')
        week_number = data.get('week_number')
        exercises = data.get('exercises')  # List of {exercise_name, target_weight}
        ########## FIX TIMEZONE ##########
        eastern = pytz.timezone('America/New_York')
        current_date = datetime.now(eastern).strftime('%Y-%m-%d')
        
        with engine.begin() as conn:
            for exercise in exercises:
                # Insert or update the plan
                conn.execute(text("""
                    INSERT INTO tenrm_plans (workout_type, week_number, exercise_name, target_weight, created_date)
                    VALUES (:workout_type, :week_number, :exercise_name, :target_weight, :created_date)
                    ON CONFLICT(workout_type, week_number, exercise_name) 
                    DO UPDATE SET target_weight = :target_weight
                """), {
                    'workout_type': workout_type,
                    'week_number': week_number,
                    'exercise_name': exercise['exercise_name'],
                    'target_weight': exercise['target_weight'],
                    'created_date': current_date
                })
        
        return jsonify({'status': 'success', 'message': '10RM plan added successfully'})
    
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e)}), 500

In [ ]:
### Logging 10 RM exercises I've completed from the plan page
@app.route('/log_10rm_completion', methods=['POST'])
def log_10rm_completion():
    """Log completion of a 10RM workout"""
    try:
        data = request.json
        plan_id = data.get('plan_id')
        completion_date = data.get('completion_date')
        completed = data.get('completed')
        notes = data.get('notes', '')
        
        ########## FIX TIMEZONE ##########
        today = datetime.now(tz=ZoneInfo("US/Eastern")).replace(microsecond=0)
        
        with engine.begin() as conn:
            conn.execute(text("""
                INSERT INTO tenrm_completions (plan_id, completion_date, completed, notes, timestamp)
                VALUES (:plan_id, :completion_date, :completed, :notes, :timestamp)"""), 
            {
                'plan_id': plan_id,
                'completion_date': completion_date,
                'completed': completed,
                'notes': notes,
                'timestamp': today
            })
        
        return jsonify({'status': 'success', 'message': 'Completion logged successfully'})
    
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e)}), 500


# Bike .Fit files

### Creating the bike workout sql tables initally

In [9]:
# A foreign key is a "pointer" from one table to another. You put the pointer in the table that needs to look up information elsewhere.
import pandas as pd
from sqlalchemy import create_engine, text 

engine = create_engine(f"sqlite:///habits.db")
with engine.begin() as conn:
    # Bike Workout table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS bike_workout (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        start_time TEXT,
                        end_time TEXT)"""))
    
    # Bike Units table (reference table - no foreign keys needed)
    conn.execute(text("""CREATE TABLE IF NOT EXISTS bike_units (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        metric_type TEXT UNIQUE NOT NULL,
                        metric_unit TEXT NOT NULL,
                        unit_name TEXT NOT NULL)"""))
    
    # Bike Workout Metrics table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS bike_metrics (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        timestamp TEXT NOT NULL,
                        workout_id INTEGER NOT NULL,
                        metric_id INTEGER NOT NULL,
                        value REAL,
                        FOREIGN KEY (workout_id) REFERENCES bike_workout(id),
                        FOREIGN KEY (metric_id) REFERENCES bike_units(id))"""))

## Reading in bike fit files

In [2]:
# Dynmaically grabbing all fit files
import os
from fitparse import FitFile

directory = r"bike_files"

bike_files = os.listdir(directory)

for file_path in bike_files:
    bike_fit_file = os.path.join(directory, file_path)
    fitfile = FitFile(bike_fit_file)

In [3]:
from fitparse import FitFile
import pandas as pd
from sqlalchemy import create_engine, text 

#fitfile = FitFile(r'bike_files/4lLGvxccOMWBlQZJ7ckLGTUQrYmXybmDNtytXseC.fit')

rows = []

# Iterate over all record messages
for record in fitfile.get_messages('record'):
    row = {}

    # Each record contains multiple data fields
    for field in record:
        row[field.name] = field.value

    rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

raw_bike_data = df.drop(columns="heart_rate")

#### Looking at units df

In [ ]:
# How we handle metric units (later)
rows = []

for record in fitfile.get_messages('record'):
    row = {}

    for field in record:
        row[field.name] = field.value
        if field.units:
            row[f"{field.name}_units"] = field.units

    rows.append(row)

df = pd.DataFrame(rows)

## Inserts done for each file

### bike_workout table

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text 

engine = create_engine(f"sqlite:///habits.db")

# Grabbing min and max times for the bike workout
max_time = max(raw_bike_data["timestamp"])
min_time = min(raw_bike_data["timestamp"])


engine = create_engine(f"sqlite:///habits.db")
with engine.begin() as conn:
    # Bike Workout table
    result = conn.execute(text("""INSERT INTO bike_workout (start_time, end_time)
                        VALUES (:start_time, :end_time)"""),
                        {"start_time": str(min_time), "end_time": str(max_time)})
    
    workout_id = result.lastrowid  # Get the ID of the workout we just inserted

### bike_units table

Mappings is something I should go back and dynamcially fill

In [ ]:
# Data cleaning step to conver the data to long format

# Statically Defining the metric-to-unit mappings
metric_unit_map = {
    'altitude': 'meters',
    'cadence': 'revolutions_per_minute',
    'distance': 'meters',
    'enhanced_altitude': 'meters',
    'enhanced_speed': 'meters_per_second',
    'position_lat': 'degrees',
    'position_long': 'degrees',
    'power': 'watts',
    'speed': 'meters_per_second'
}

# Statically defining the unit to abbreviation mappings
unit_abv_map = {
    'meters': 'm',
    'revolutions_per_minute': 'rpm',
    'meters_per_second': 'm/s',
    'degrees': 'deg',
    'watts': 'W',
}

# Melt the dataframe
bike_data_long = pd.melt(raw_bike_data, 
              id_vars=['timestamp'], 
              value_vars=['altitude', 'cadence', 'distance', 'enhanced_altitude',
                         'enhanced_speed', 'position_lat', 'position_long',
                         'power', 'speed'], 
              var_name="metric", 
              value_name="value")


# Add unit columns - no lambda needed!
bike_data_long['unit_name'] = bike_data_long['metric'].map(metric_unit_map) 
bike_data_long['metric_unit'] = bike_data_long['unit_name'].map(unit_abv_map)

In [ ]:
# Uploading data to the bike units df only if it's new 

# Filtering bike_data_long to only revlavent columns associated with bike_units table
bike_units_df = bike_data_long[["metric", "unit_name", "metric_unit"]].drop_duplicates() # drop_dups to get unique values

engine = create_engine(f"sqlite:///habits.db")

for _, row in bike_units_df.iterrows():
    metric_type = row["metric"]
    metric_unit = row["metric_unit"]
    unit_name = row["unit_name"]

    with engine.begin() as conn:
        # Insert only if it doesn't already exist
        conn.execute(text("""
            INSERT OR IGNORE INTO bike_units (metric_type, metric_unit, unit_name)
            VALUES (:metric_type, :metric_unit, :unit_name)"""), 
            {"metric_type": metric_type, "metric_unit": metric_unit, "unit_name": unit_name})

### bike_metrics table

In [7]:
# Selecting data only relavent to bike_metrics table insertion 
bike_metrics_df = bike_data_long[["timestamp", "metric", "value"]].drop_duplicates().copy()

engine = create_engine(f"sqlite:///habits.db")

# Grabbing associated workout_id (usually coming from bike_workout insertion code)
#with engine.begin() as conn:
#    bike_workout = pd.read_sql_query(text("""SELECT id 
#                                        FROM bike_workout"""), conn)
#bike_workout_id = int(bike_workout["id"].values[0]) 

# workout_id comes from above insertion code
bike_workout_id = workout_id

# Grabbing all the metric types from the metrics table which I'll use to filter incoming bike data and insert into bike_metrics
with engine.begin() as conn:
    bike_units = pd.read_sql_query(text("""SELECT id, metric_type 
                                        FROM bike_units"""), conn)

bike_metrics_df["workout_id"] = bike_workout_id

In [ ]:
# Looping through each bike metric and uploading data from new bike file with the appropriate bike metric id 
for _, row in bike_units.iterrows():
   metric_name = row["metric_type"]
   
   bike_metrics_upload_df = bike_metrics_df[bike_metrics_df["metric"] == metric_name].copy()

   # Now that we're filtered by metric name we can just insert the id for that metric
   bike_metrics_upload_df["metric_id"] = row["id"]
   
   # After filtering by metric column droppping it because now we have it's ID!
   bike_metrics_upload_df.drop(columns=["metric"], inplace=True)

   # Appending the new raw data to the apple_data_raw table
   bike_metrics_upload_df.to_sql(name="bike_metrics", con=engine, if_exists='append', index=False)